In [1]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForTokenClassification
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel
from data_loader import *
from datasets import Dataset, DatasetDict

In [2]:
num_input_chars=12

In [3]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

In [4]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=1024,
    num_hidden_layers=3,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.0,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False,
    label_smoothing_factor=0.1,
)


In [5]:
df, labels = load_ner_dataset()
labels = list(range(max(labels) + 1))


df['text'] = df['tokens'].apply(lambda x: ' '.join(x))
# remove empty text rows
df = df[df['text'].str.strip().astype(bool)].sample(frac=1.0, random_state=42).reset_index(drop=True)

README.md: 0.00B [00:00, ?B/s]

masakhaner2.py: 0.00B [00:00, ?B/s]

bam/train/0000.parquet:   0%|          | 0.00/349k [00:00<?, ?B/s]

bam/validation/0000.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

bam/test/0000.parquet:   0%|          | 0.00/106k [00:00<?, ?B/s]

...

zul/test/0000.parquet:   0%|          | 0.00/162k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5848 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/836 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1670 [00:00<?, ? examples/s]

Loaded 152786 rows from masakhanener columns Index(['id', 'tokens', 'labels', 'lang', 'split'], dtype='object')


In [6]:
df

,id,tokens,labels,lang,split,text
0,3002,"[Aliyekuwa, rais, wa, Burundi, Pierre, Buyoya,...","[0, 0, 0, 5, 1, 2, 0, 0, 7, 0, 0, 0, 0, 5, 0, ...",swa,train,Aliyekuwa rais wa Burundi Pierre Buyoya amezik...
1,1360,"[Doctor, Sikosana, vati, nekudaro, hurumende, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sna,test,Doctor Sikosana vati nekudaro hurumende yave k...
2,2657,"[Iye, adati, kupatula, kusemphana, maganizoku,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",nya,train,"Iye adati kupatula kusemphana maganizoku , nkh..."
3,1999,"[Anotaura, akamirira, veZimbabwe, Football, As...","[0, 0, 3, 4, 4, 0, 1, 2, 0, 0, 0, 0, 0, 5, 0, ...",sna,train,Anotaura akamirira veZimbabwe Football Associa...
4,619,"[To, develop, broad, -, based, mechanism, and,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",pcm,test,To develop broad - based mechanism and trainin...
...,...,...,...,...,...,...
152781,985,"[Typewriter, -, no, koro, biro, bedo, achiel, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",luo,test,Typewriter - no koro biro bedo achiel kuom gik...
152782,2578,"[Nathi, siwumphakathi, masisebenzisane, namaph...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",zul,train,Nathi siwumphakathi masisebenzisane namaphoyis...
152783,1009,"[Waaye, nettali, bii, dafay, joxe, yenn, digle...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",wol,test,"Waaye nettali bii dafay joxe yenn digle , bu d..."
152784,306,"[Zvichakadaro, ,, Insurance, Pension's, Commis...","[0, 0, 3, 4, 4, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, ...",sna,dev,"Zvichakadaro , Insurance Pension's Commission ..."


In [7]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

Dev split has 20 languages.
Test split has 20 languages.
Train split has 20 languages.


In [8]:
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")
print(f"Converted integers to labels: {id2label}")

Converted labels to integers: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8}
Converted integers to labels: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8}


In [9]:
config.num_labels = len(label2id)
model = BertForTokenClassification(config)

In [10]:
# model.save_pretrained("ner-model")
# tokenizer.save_pretrained("ner-model")

In [11]:
# model.from_pretrained("ner-model")

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): DistilEmbeddings(
      (word_embeddings): DistillEmb(
        (encoder): DistillEmbBase(
          (embedding): Embedding(1518, 128)
          (conv1): Conv1d(12, 128, kernel_size=(5,), stride=(1,))
          (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,))
          (conv3): Conv1d(256, 384, kernel_size=(5,), stride=(1,))
          (conv4): Conv1d(384, 448, kernel_size=(3,), stride=(1,))
          (conv5): Conv1d(448, 512, kernel_size=(3,), stride=(1,))
          (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (output_layer): Linear(in_features=512, out_features=512, bias=True)
          (activation): GELU(approximate='none')
          (norm0): LayerNorm((12, 128), eps=1e-05, elementwise_affine=True)
          (norm1): LayerNorm((128, 62), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256, 29), eps=1e-05, elementwise_affine=True)
          (norm3

In [12]:

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

In [13]:

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

Dataset({
    features: ['id', 'tokens', 'labels', 'lang', 'split', 'text', '__index_level_0__'],
    num_rows: 106964
})

In [14]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=512,
        return_attention_mask=False,
    )

    batch["labels"] = examples["labels"]
    return batch

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/106964 [00:00<?, ? examples/s]

Map:   0%|          | 0/30538 [00:00<?, ? examples/s]

In [15]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
        )
        
        max_len = batch["input_ids"].shape[1] - 2  # exclude special tokens
        padded_labels = []
        for f in features:
            label = f["labels"]
            
            padded_label = [-100] +  label + [-100] * (max_len - len(label)) + [-100]
            padded_labels.append(padded_label)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        
        assert batch["labels"].shape == (batch["input_ids"].shape[0], batch["input_ids"].shape[1]), f"Labels shape {batch['labels'].shape} does not match input_ids shape {batch['input_ids'].shape}"
        return batch

data_collator = CustomDataCollator(tokenizer)

In [ ]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        mask = label_seq != -100
        true_labels.extend(label_seq[mask])
        pred_labels.extend(pred_seq[mask])

    label_ids = list(label2id.values())

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "f1_weighted": f1_score(true_labels, pred_labels, average="weighted", labels=label_ids, zero_division=0),
        "f1_macro": f1_score(true_labels, pred_labels, average="macro", labels=label_ids, zero_division=0),
        "f1_micro": f1_score(true_labels, pred_labels, average="micro", labels=label_ids, zero_division=0),
    }


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 16

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=20,
    weight_decay=0.0,
    label_smoothing_factor=0.1,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=16,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro,F1 Micro
1,0.615600,0.575953,0.962193,0.960681,0.788107,0.962193


In [ ]:
labels

In [ ]:
from collections import defaultdict

pred_output = trainer.predict(tokenized_test)
logits = pred_output.predictions
label_ids = pred_output.label_ids
langs = test_df["lang"].tolist()

lang_true, lang_pred = defaultdict(list), defaultdict(list)
for idx, lang in enumerate(langs):
    label_seq = label_ids[idx]
    pred_seq = logits[idx].argmax(axis=-1)
    mask = label_seq != -100
    if not np.any(mask):
        continue
    lang_true[lang].extend(label_seq[mask])
    lang_pred[lang].extend(pred_seq[mask])

label_id_list = list(label2id.values())
lang_metrics = {}
for lang, true_values in lang_true.items():
    preds = lang_pred[lang]
    lang_metrics[lang] = {
        "accuracy": accuracy_score(true_values, preds),
        "f1_weighted": f1_score(true_values, preds, average="weighted", labels=label_id_list, zero_division=0),
        "f1_macro": f1_score(true_values, preds, average="macro", labels=label_id_list, zero_division=0),
        "f1_micro": f1_score(true_values, preds, average="micro", labels=label_id_list, zero_division=0),
        "num_tokens": len(true_values),
    }

lang_metrics_df = pd.DataFrame.from_dict(lang_metrics, orient="index").sort_values("accuracy", ascending=False)

lang_metrics_df

In [ ]:
print(lang_metrics_df['accuracy'].mean())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(lang_metrics_df["num_tokens"], lang_metrics_df["accuracy"], alpha=0.7)
ax.set_xlabel("Number of Tokens")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs. Number of Tokens by Language")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
plt.show()

In [1]:
# 82.2 74.8 90.3 82.7 87.4 89.6 87.5 89.6 82.2 76.4 92.4 89.7 96.2 92.7 89.4 81.1 86.8 89.9 89.3 90.6
mean = sum([82.2, 74.8, 90.3, 82.7, 87.4, 89.6, 87.5, 89.6, 82.2, 76.4, 92.4, 89.7, 96.2, 92.7, 89.4, 81.1, 86.8, 89.9, 89.3, 90.6]) / 20
mean

87.03999999999999

In [2]:
from datasets import load_dataset
data = load_dataset('masakhane/masakhaner2', 'yor') 

# Please, specify the language code

# # A data point consists of sentences seperated by empty line and tab-seperated tokens and tags. 
# {'id': '0',
#  'ner_tags': [B-DATE, I-DATE, 0, 0, 0, 0, 0, B-PER, I-PER, I-PER, O, O, O, O],
#  'tokens': ['Wákàtí', 'méje', 'ti', 'ré', 'kọjá', 'lọ', 'tí', 'Luis', 'Carlos', 'Díaz', 'ti', 'di', 'awati', '.']
# }


Using the latest cached version of the dataset since masakhane/masakhaner2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'yor' at /home/leo/.cache/huggingface/datasets/masakhane___masakhaner2/yor/1.0.0/60512e89e68841b6b5ed1be59caf97b169f0d27a (last modified on Mon Dec  8 10:08:23 2025).
